# Wilton Manor Data Cleaning Pipeline
## City-Specific Data Normalization

This notebook handles **Wilton Manor specific** data cleaning and normalization:
- Loads raw JSON files from `results_folder/wiltonmanor/raw_json/`
- Extracts tables from JSON chunks with proper header handling
- Applies Wilton Manor-specific field mappings
- Outputs clean, normalized data ready for financial analysis

**Input**: Raw JSON files from extraction pipeline  
**Output**: Clean, normalized CSV ready for joining with other cities

## Environment Setup

In [2]:
import pandas as pd
import json
from pathlib import Path
from io import StringIO
import sys

# Add parent directory to path for imports
sys.path.append(str(Path.cwd().parent))

# Import functions directly to avoid relative import issues
try:
    from normalizers import normalize_wilton
    print("Successfully imported normalize_wilton")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Let's import the functions we need directly...")
    
    # If normalizers import fails, we'll define a simple version here
    def normalize_wilton(df, city, source_file=None):
        """Simple Wilton Manor normalizer"""
        rename_map = {
            "File#": "violation_id_raw",
            "Address": "address_raw",
            "Violation": "violation_description_raw",
            "Open Date": "opened_date_raw",
            "Status": "case_status_raw"
        }
        df2 = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}).copy()
        df2["city"] = city
        df2["source_file"] = source_file
        return df2

# Path masking function
def mask_path(p: str | Path) -> str:
    p = Path(p)
    parts = p.parts
    return str(Path(*(["..."] + list(parts[-3:]))))

# Project paths
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[2]
else:
    cwd = Path.cwd()
    if cwd.name == "cleaning":
        ROOT = cwd.parents[1]
    elif cwd.name == "src":
        ROOT = cwd.parent
    else:
        ROOT = cwd

RESULTS_DIR = ROOT / "results_folder"
WILTON_DIR = RESULTS_DIR / "wiltonmanor"
CLEAN_DIR = ROOT / "clean_data"
CLEAN_DIR.mkdir(exist_ok=True)

print("ROOT:", mask_path(ROOT))
print("WILTON_DIR:", mask_path(WILTON_DIR))
print("CLEAN_DIR:", mask_path(CLEAN_DIR))

# Find JSON files
json_dir = WILTON_DIR / "raw_json"
if json_dir.exists():
    json_files = list(json_dir.glob("*.json"))
    print(f"Found {len(json_files)} JSON files")
    for f in json_files:
        print(f"  - {f.name}")
else:
    print(f"❌ JSON directory not found: {mask_path(json_dir)}")

❌ Import error: attempted relative import with no known parent package
Let's import the functions we need directly...
ROOT: ...\Edilma Projects\LandingAI-Hack\coderisk-sf
WILTON_DIR: ...\coderisk-sf\results_folder\wiltonmanor
CLEAN_DIR: ...\LandingAI-Hack\coderisk-sf\clean_data
Found 1 JSON files
  - wilton_Code_Violation_to_March_2024-30p.json


## Extract Tables from JSON Chunks

In [ ]:
def extract_wilton_tables(json_file_path):
    """
    Extract tables from Wilton Manor JSON chunks with mixed header handling.
    First table has headers in row 0, subsequent tables are data-only.
    """
    print(f"Processing Wilton Manor JSON: {Path(json_file_path).name}")
    
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_tables = []
    master_headers = None
    
    if 'chunks' in data:
        for i, chunk in enumerate(data['chunks']):
            if chunk.get('type') == 'table' and 'markdown' in chunk:
                print(f"  Processing table chunk {i+1}")
                
                df = parse_html_table_mixed_headers(chunk['markdown'], master_headers, i == 0)
                if df is not None and not df.empty:
                    # Store headers from first table
                    if i == 0 and master_headers is None:
                        master_headers = list(df.columns)
                        print(f"    Master headers established: {master_headers}")
                    
                    # Add metadata
                    df['chunk_id'] = i
                    df['source_file'] = mask_path(json_file_path)
                    all_tables.append(df)
                    print(f"    Extracted {len(df)} rows")
    
    if all_tables:
        combined_df = pd.concat(all_tables, ignore_index=True)
        print(f"Total extracted: {len(combined_df)} rows")
        return combined_df
    else:
        print("❌ No table chunks found")
        return pd.DataFrame()

def parse_html_table_mixed_headers(html_content, master_headers=None, is_first_table=False):
    """Parse HTML table handling mixed header situation for Wilton Manor"""
    try:
        if '<table' in html_content.lower():
            tables = pd.read_html(StringIO(html_content), header=None)
            if tables:
                df = tables[0]
                
                if is_first_table:
                    # First table: extract headers from row 0
                    if len(df) > 1:
                        headers = df.iloc[0].fillna('').astype(str).tolist()
                        data_df = df.iloc[1:].reset_index(drop=True)
                        data_df.columns = headers[:len(data_df.columns)]
                        return data_df
                else:
                    # Subsequent tables: use master headers, all rows are data
                    if master_headers and len(master_headers) >= len(df.columns):
                        df.columns = master_headers[:len(df.columns)]
                        return df
                
                return df
    except Exception as e:
        print(f"    ❌ Parse error: {e}")
    return None

## Process All Wilton Manor Files

In [4]:
# Process all Wilton Manor JSON files
print("Processing ALL Wilton Manor files...")

all_data = []

for i, json_file in enumerate(json_files, 1):
    print(f"\n[{i}/{len(json_files)}] Processing: {json_file.name}")
    df = extract_wilton_tables(json_file)
    if not df.empty:
        all_data.append(df)
        print(f"  Got {len(df)} rows")
    else:
        print(f"  ❌ No data from this file")

if all_data:
    # Combine all data
    wilton_raw = pd.concat(all_data, ignore_index=True)
    
    print(f"\nRAW WILTON MANOR DATA:")
    print(f"Shape: {wilton_raw.shape}")
    print(f"Columns: {list(wilton_raw.columns)}")
    print(f"\nSample data:")
    print(wilton_raw.head(3))
    
else:
    print("❌ No data extracted from any files")
    wilton_raw = pd.DataFrame()

Processing ALL Wilton Manor files...

[1/1] Processing: wilton_Code_Violation_to_March_2024-30p.json
Processing Wilton Manor JSON: wilton_Code_Violation_to_March_2024-30p.json
  Processing table chunk 5
    Extracted 16 rows
  Processing table chunk 10
    Extracted 18 rows
  Processing table chunk 15
    Extracted 18 rows
  Processing table chunk 20
    Extracted 18 rows
  Processing table chunk 25
    Extracted 18 rows
  Processing table chunk 30
    Extracted 18 rows
  Processing table chunk 35
    Extracted 18 rows
  Processing table chunk 40
    Extracted 18 rows
  Processing table chunk 45
    Extracted 18 rows
  Processing table chunk 50
    Extracted 18 rows
  Processing table chunk 55
    Extracted 18 rows
  Processing table chunk 60
    Extracted 18 rows
  Processing table chunk 65
    Extracted 18 rows
  Processing table chunk 70
    Extracted 18 rows
  Processing table chunk 75
    Extracted 18 rows
  Processing table chunk 80
    Extracted 18 rows
  Processing table chunk 

## Data Cleaning

In [5]:
if not wilton_raw.empty:
    print("Cleaning Wilton Manor data...")
    
    wilton_clean = wilton_raw.copy()
    
    print(f"Before cleaning: {len(wilton_clean)} rows")
    
    # Remove any obvious header repeats
    if "File#" in wilton_clean.columns:
        header_mask = wilton_clean["File#"].astype(str).str.strip() == "File#"
        rows_before = len(wilton_clean)
        wilton_clean = wilton_clean[~header_mask]
        print(f"Removed {rows_before - len(wilton_clean)} header repeat rows")
    
    # Remove empty rows
    rows_before = len(wilton_clean)
    wilton_clean = wilton_clean.dropna(how='all')
    print(f"Removed {rows_before - len(wilton_clean)} completely empty rows")
    
    print(f"After cleaning: {len(wilton_clean)} rows")
    
    # Show data overview
    print(f"\nColumn overview:")
    for col in wilton_clean.columns:
        non_null = wilton_clean[col].notna().sum()
        print(f"  {col}: {non_null} non-null values")
    
    print(f"\nSample cleaned data:")
    print(wilton_clean.head(3))
    
else:
    print("❌ No raw data to clean")
    wilton_clean = pd.DataFrame()

Cleaning Wilton Manor data...
Before cleaning: 538 rows
Removed 0 header repeat rows
Removed 0 completely empty rows
After cleaning: 538 rows

Column overview:
  File#: 16 non-null values
  Violation: 16 non-null values
  Address: 16 non-null values
  Open Date: 16 non-null values
  Status: 16 non-null values
  chunk_id: 538 non-null values
  source_file: 538 non-null values
  24-000007: 18 non-null values
  Building Safety Inspection Program.: 108 non-null values
  2607 NE 8 Avenue # 27: 18 non-null values
  01/02/2024: 18 non-null values
  Closed: 342 non-null values
  24-000030: 18 non-null values
  2607 NE 8 Avenue # 43: 18 non-null values
  01/03/2024: 36 non-null values
  24-000025: 18 non-null values
  2607 NE 8 Avenue # 39: 18 non-null values
  24-000073: 18 non-null values
  1951 NE 2 Avenue UNIT 104-I: 18 non-null values
  01/08/2024: 18 non-null values
  24-000130: 18 non-null values
  1940 NE 2 Avenue UNIT 108-J: 18 non-null values
  01/09/2024: 18 non-null values
  24-0001

## Apply Normalization

In [6]:
if not wilton_clean.empty:
    print("Applying Wilton Manor normalization...")
    
    # Apply the normalize_wilton function
    wilton_normalized = normalize_wilton(
        wilton_clean, 
        city="Wilton Manor", 
        source_file="wilton_Code_Violation_to_March_2024-30p.json"
    )
    
    print(f"Normalization complete!")
    print(f"Shape: {wilton_normalized.shape}")
    print(f"Columns: {list(wilton_normalized.columns)}")
    
    # Show sample normalized data
    print(f"\nSample normalized data:")
    key_cols = ['violation_id_raw', 'address_raw', 'case_status_raw', 'opened_date_raw']
    display_cols = [col for col in key_cols if col in wilton_normalized.columns]
    print(wilton_normalized[display_cols].head(5))
    
    # Show unique case count
    if 'violation_id_raw' in wilton_normalized.columns:
        unique_cases = wilton_normalized['violation_id_raw'].nunique()
        print(f"\nFound {unique_cases} unique cases")
    
else:
    print("❌ No clean data to normalize")
    wilton_normalized = pd.DataFrame()

Applying Wilton Manor normalization...
Normalization complete!
Shape: (538, 109)
Columns: ['violation_id_raw', 'violation_description_raw', 'address_raw', 'opened_date_raw', 'case_status_raw', 'chunk_id', 'source_file', '24-000007', 'Building Safety Inspection Program.', '2607 NE 8 Avenue # 27', '01/02/2024', 'Closed', '24-000030', '2607 NE 8 Avenue # 43', '01/03/2024', '24-000025', '2607 NE 8 Avenue # 39', '24-000073', '1951 NE 2 Avenue UNIT 104-I', '01/08/2024', '24-000130', '1940 NE 2 Avenue UNIT 108-J', '01/09/2024', '24-000138', '1940 NE 2 Avenue UNIT 112-J', '01/10/2024', '24-000201', 'Screens, shutters and awnings.', '2312 WILTON Drive', '01/16/2024', 'Open', '24-000251', 'Local Business Tax Receipts', '2809 NW 7 Avenue', '01/22/2024', '24-000286', '2205 WILTON Drive', '01/24/2024', '24-000301', '2740 North Andrews Avenue', '01/29/2024', '24-000338', 'Container storage, enclosure and screening.', '200 NW 25 Street', '01/30/2024', '24-000185', 'Required Permit Approvals - buildin

## Save Results

In [7]:
if not wilton_normalized.empty:
    # Save cleaned data
    output_file = CLEAN_DIR / "wilton_manor_clean.csv"
    wilton_normalized.to_csv(output_file, index=False)
    
    print(f"Saved to: {mask_path(output_file)}")
    
    # Final stats
    print(f"\nWILTON MANOR RESULTS:")
    print(f"  - Total rows: {len(wilton_normalized)}")
    if 'violation_id_raw' in wilton_normalized.columns:
        print(f"  - Unique cases: {wilton_normalized['violation_id_raw'].nunique()}")
    if 'case_status_raw' in wilton_normalized.columns:
        print(f"  - Status breakdown:")
        status_counts = wilton_normalized['case_status_raw'].value_counts()
        for status, count in status_counts.head(5).items():
            print(f"    {status}: {count}")
    
    # Date range if available
    if 'opened_date_raw' in wilton_normalized.columns:
        date_col = pd.to_datetime(wilton_normalized['opened_date_raw'], errors='coerce')
        valid_dates = date_col.dropna()
        if not valid_dates.empty:
            print(f"  - Date range: {valid_dates.min().date()} to {valid_dates.max().date()}")
    
    print(f"\nSUCCESS! Wilton Manor data ready for financial analysis!")
    
else:
    print("❌ No data to save")

Saved to: ...\coderisk-sf\clean_data\wilton_manor_clean.csv

WILTON MANOR RESULTS:
  - Total rows: 538
  - Unique cases: 16
  - Status breakdown:
    Closed: 16
  - Date range: 2024-01-02 to 2024-01-02

SUCCESS! Wilton Manor data ready for financial analysis!
